<a href="https://colab.research.google.com/github/Aldiss1/MPPM_Aldi/blob/main/JS03/Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Muat dataset Wisconsin Breast Cancer
cancer = load_breast_cancer()
df_cancer = pd.DataFrame(data=cancer.data, columns=cancer.feature_names)
df_cancer['diagnosis'] = cancer.target

# Pisahkan fitur (X) dan target (y)
X = df_cancer.drop('diagnosis', axis=1)
y = df_cancer['diagnosis']

# Tampilkan 5 baris pertama data dan statistik deskriptif
print("\n--- 5 Baris Pertama Data --- ")
display(df_cancer.head())
print("\n--- Statistik Deskriptif --- ")
display(df_cancer.describe())


--- 5 Baris Pertama Data --- 


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0



--- Statistik Deskriptif --- 


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,0.062798,...,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946,0.627417
std,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,0.007060,...,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061,0.483918
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,0.049960,...,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040,0.000000
25%,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,0.057700,...,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460,0.000000
50%,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,0.061540,...,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040,1.000000
75%,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,0.066120,...,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080,1.000000
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,0.097440,...,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500,1.000000


Data 'diagnosis' sudah berupa numerik (0 dan 1), jadi tidak perlu encoding tambahan. Semua fitur input adalah numerik.



In [45]:
# Identifikasi kolom numerik (semua fitur di dataset ini adalah numerik)
num_features = X.columns.tolist()

# Buat transformer untuk data numerik
# Imputer untuk menangani missing value (jika ada), lalu scaler
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Buat preprocessor menggunakan ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features)
    ])

# Bagi data menjadi training dan testing set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [46]:
# Buat pipeline lengkap
# k='all' pada SelectKBest awalnya untuk melihat semua skor fitur, lalu akan dicari nilai k terbaik
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('selector', SelectKBest(score_func=f_classif, k='all')),
    ('classifier', LogisticRegression(random_state=42, solver='liblinear'))
])

# Latih model
model_pipeline.fit(X_train, y_train)

# Prediksi pada test set
y_pred = model_pipeline.predict(X_test)

# Evaluasi model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.4f}")
print("\nClassification Report:\n", report)


Accuracy: 0.9737

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.95      0.96        43
           1       0.97      0.99      0.98        71

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



In [47]:
# Dapatkan skor fitur dari SelectKBest
feature_scores = model_pipeline.named_steps['selector'].scores_
feature_names = X.columns

# Buat DataFrame dari skor fitur
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Score': feature_scores
})

# Urutkan fitur berdasarkan skor
feature_importance_df = feature_importance_df.sort_values(by='Score', ascending=False)

print("\n--- Skor Fitur dari SelectKBest (diurutkan) --- ")
display(feature_importance_df)

k_best = 10 # Contoh: Memilih 10 fitur terbaik

model_pipeline_k_best = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('selector', SelectKBest(score_func=f_classif, k=k_best)),
    ('classifier', LogisticRegression(random_state=42, solver='liblinear'))
])

model_pipeline_k_best.fit(X_train, y_train)

y_pred_k_best = model_pipeline_k_best.predict(X_test)
accuracy_k_best = accuracy_score(y_test, y_pred_k_best)
report_k_best = classification_report(y_test, y_pred_k_best)

print(f"\nAccuracy dengan {k_best} fitur terbaik: {accuracy_k_best:.4f}")
print("\nClassification Report dengan 10 fitur terbaik:\n", report_k_best)

# Dapatkan nama fitur yang terpilih
selected_features_mask = model_pipeline_k_best.named_steps['selector'].get_support()
selected_feature_names = feature_names[selected_features_mask].tolist()

print(f"\nJumlah fitur terbaik yang dapat digunakan (k={k_best}): {len(selected_feature_names)}")
print("Fitur-fitur tersebut adalah:")
for feature in selected_feature_names:
    print(f"- {feature}")


--- Skor Fitur dari SelectKBest (diurutkan) --- 


,Feature,Score
27,worst concave points,746.492117
7,mean concave points,695.179785
22,worst perimeter,681.263759
20,worst radius,645.350668
2,mean perimeter,522.489267
23,worst area,495.787667
0,mean radius,482.233945
3,mean area,423.654133
6,mean concavity,396.662370
26,worst concavity,331.330906



Accuracy dengan 10 fitur terbaik: 0.9737

Classification Report dengan 10 fitur terbaik:
               precision    recall  f1-score   support

           0       0.95      0.98      0.97        43
           1       0.99      0.97      0.98        71

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114


Jumlah fitur terbaik yang dapat digunakan (k=10): 10
Fitur-fitur tersebut adalah:
- mean radius
- mean perimeter
- mean area
- mean concavity
- mean concave points
- worst radius
- worst perimeter
- worst area
- worst concavity
- worst concave points
